# SkinGen: Ingredient-Aware Recommendation System

This notebook extends the concern-based recommender with **ingredient-level intelligence**.

## Key Enhancements

**Previous system**: Filters by product-level concern labels (e.g., "hydrating", "irritating")

**This system**:
1. **Ingredient intelligence scoring**: Analyzes 4,985 ingredients with properties (categories, benefits, functional groups)
2. **Required/blocked ingredient filtering**: User can specify "must have niacinamide" or "no fragrance"
3. **Comprehensive safety metric**: Includes both label-based AND ingredient-based penalties
4. **Explainability**: Shows WHY each product was recommended

## Hypothesis

Ingredient intelligence should improve:
- **Relevance**: Products with actual active ingredients rank higher
- **Safety**: Products with irritants/fragrance automatically penalized
- **Trustworthiness**: Less reliance on potentially misleading product labels

## Setup

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import ndcg_score
from sklearn.model_selection import train_test_split, KFold
import json

# Load data
products = pd.read_parquet('../data/cleaned/skingen_products_lean_clean.parquet')
ingredients_db = pd.read_parquet('../data/cleaned/ingredients_enriched.parquet')

with open('../data/synthetic_test_queries.json', 'r') as f:
    all_queries = json.load(f)

print(f'Loaded {len(products):,} products')
print(f'Loaded {len(ingredients_db):,} ingredient mappings')
print(f'Loaded {len(all_queries)} test queries')

Loaded 10,296 products
Loaded 4,985 ingredient mappings
Loaded 200 test queries


## Part 1: Base Recommender Setup

In [2]:
# Fix NaN arrays
def fix_nan(arr):
    return arr if isinstance(arr, np.ndarray) else np.array([])

products['positive_concerns'] = products['positive_concerns'].apply(fix_nan)
products['negative_concerns'] = products['negative_concerns'].apply(fix_nan)
products['condition_concerns'] = products['condition_concerns'].apply(fix_nan)

# One-hot encoding
mlb_pos = MultiLabelBinarizer()
mlb_neg = MultiLabelBinarizer()
mlb_cond = MultiLabelBinarizer()

pos_encoded = mlb_pos.fit_transform(products['positive_concerns'])
neg_encoded = mlb_neg.fit_transform(products['negative_concerns'])
cond_encoded = mlb_cond.fit_transform(products['condition_concerns'])

# Build features
features = np.hstack([pos_encoded, neg_encoded, cond_encoded])

print(f'Feature matrix: {features.shape}')

Feature matrix: (10296, 17)


In [3]:
# Safety rules
CONCERN_RULES = {
    'redness_reducing': {'irritating'},
    'pore_minimizing': {'acne_trigger'},
    'good_for_acne_prone_skin': {'acne_trigger'}
}

SKIN_TYPE_PENALTY_RULES = {
    'dry_skin': ['drying'],
    'oily_skin': ['may_worsen_oily_skin'],
    'sensitive_skin': ['irritating', 'drying'],
    'combination_skin': ['drying', 'may_worsen_oily_skin']
}

def create_query_vector(user_query):
    """Create feature vector from user query"""
    query = np.zeros(features.shape[1])
    user_concerns = user_query.get('concerns', [])
    
    if user_concerns:
        valid = [c for c in user_concerns if c in mlb_pos.classes_]
        if valid:
            pos_vec = mlb_pos.transform([valid])[0]
            query[0:len(mlb_pos.classes_)] = pos_vec
    
    return query

print('✅ Base recommender setup complete')

✅ Base recommender setup complete


## Part 2: Ingredient Intelligence Mapping

In [4]:
# User Concerns → Ingredient Categories (99.3% coverage)
CONCERN_TO_CATEGORY = {
    'hydrating': ['Humectant', 'Emollient', 'Occlusive/Opacifying Agent'],
    'anti_aging': ['Antioxidant', 'Peptides', 'Retinoids'],
    'brightening': ['Antioxidant', 'Exfoliant'],
    'dark_spots': ['Antioxidant', 'Exfoliant'],
    'soothing': ['Plant Extracts', 'Antioxidant'],
    'redness_reducing': ['Plant Extracts', 'Antioxidant'],
    'exfoliating': ['Exfoliant'],
    'good_for_oily_skin': ['Absorbent'],
    'reduces_large_pores': ['Absorbent', 'Exfoliant'],
    'firming': ['Peptides', 'Film-Forming Agent'],
    'skin_texture': ['Texture Enhancer', 'Exfoliant'],
    
    # Ingredient-specific concerns (users can select these directly)
    'antioxidant': ['Antioxidant'],
    'peptides': ['Peptides'],
    'retinoids': ['Retinoids']
}

# User Concerns → Ingredient Benefits (38.9% coverage - bonus)
CONCERN_TO_BENEFIT = {
    'hydrating': ['Hydration'],
    'anti_aging': ['Anti-Aging'],
    'soothing': ['Soothing'],
    'brightening': ['Evens Skin Tone', 'Dark Spot Fading'],
    'dark_spots': ['Dark Spot Fading'],
    'good_for_oily_skin': ['Oil Control'],
    'good_for_acne_prone_skin': ['Anti-Acne'],
    'reduces_large_pores': ['Pore Minimizer'],
}

# Functional Group Weights
FUNCTIONAL_WEIGHTS = {
    'Actives': 2.0,
    'Support': 0.5,
    'Sensory': 0.25,
    'Utility': 0.5,
    'Risks': -1.0
}

# Ingredient Rating Multipliers (99.9% coverage)
RATING_MULTIPLIERS = {
    'Best': 1.3,      # 30% boost for premium ingredients
    'Good': 1.1,      # 10% boost for good quality
    'Average': 1.0,   # Neutral (no change)
    'Bad': 0.8,       # 20% penalty for problematic ingredients
    'Worst': 0.5      # 50% penalty for worst ingredients
}

# Negative categories to penalize
NEGATIVE_CATEGORIES = {'Irritant', 'Fragrance: Synthetic and Natural'}

print('✅ Ingredient mapping loaded')

✅ Ingredient mapping loaded


In [5]:
# Build ingredient lookup dictionary
ingredient_lookup = {}
for _, row in ingredients_db.iterrows():
    ingredient_lookup[row['ingredient_name'].lower()] = {
        'name': row['ingredient_name'],
        'rating': row['rating'],
        'categories': row['categories'] if isinstance(row['categories'], np.ndarray) else [],
        'benefits': row['benefits'] if isinstance(row['benefits'], np.ndarray) else [],
        'functional_group': row['functional_group'] if isinstance(row['functional_group'], np.ndarray) else []
    }

print(f'✅ Built lookup for {len(ingredient_lookup):,} ingredients')

✅ Built lookup for 4,985 ingredients


In [6]:
def calculate_ingredient_score(product, user_concerns, ingredient_db):
    """
    Score product based on ingredient intelligence
    
    Scoring components:
    1. Category match → +1.0 points per match
    2. Benefit match → +0.5 bonus
    3. Functional weight → multiply by group importance (Actives=2x, Risks=-1x)
    4. Rating multiplier → multiply by quality (Best=1.3x, Worst=0.5x)
    
    Returns:
        score: Normalized ingredient intelligence score
        has_risks: Boolean indicating if product contains risky ingredients
    """
    total_score = 0
    has_risks = False
    
    if not isinstance(product['ingredient_list'], np.ndarray):
        return 0, False
    
    for ing in product['ingredient_list']:
        ing_lower = ing.lower()
        
        if ing_lower not in ingredient_db:
            continue
        
        ing_info = ingredient_db[ing_lower]
        ingredient_value = 0
        
        # ====================================================================
        # 1. CATEGORY MATCHING (Primary scoring)
        # ====================================================================
        categories = ing_info.get('categories', [])
        
        for concern in user_concerns:
            if concern in CONCERN_TO_CATEGORY:
                target_categories = CONCERN_TO_CATEGORY[concern]
                for cat in categories:
                    if cat in target_categories:
                        ingredient_value += 1.0
                        break
        
        # Penalize negative categories
        for cat in categories:
            if cat in NEGATIVE_CATEGORIES:
                ingredient_value -= 1.0
                has_risks = True
        
        # ====================================================================
        # 2. BENEFIT MATCHING (Bonus)
        # ====================================================================
        benefits = ing_info.get('benefits', [])
        if len(benefits) > 0:
            for concern in user_concerns:
                if concern in CONCERN_TO_BENEFIT:
                    target_benefits = CONCERN_TO_BENEFIT[concern]
                    for ben in benefits:
                        if ben in target_benefits:
                            ingredient_value += 0.5
                            break
        
        # ====================================================================
        # 3. FUNCTIONAL GROUP WEIGHTING
        # ====================================================================
        functional_groups = ing_info.get('functional_group', [])
        max_weight = 1.0
        for func in functional_groups:
            if func in FUNCTIONAL_WEIGHTS:
                weight = FUNCTIONAL_WEIGHTS[func]
                if abs(weight) > abs(max_weight):
                    max_weight = weight
                if func == 'Risks':
                    has_risks = True
        
        ingredient_value *= max_weight
        
        # ====================================================================
        # 4. NEW: RATING MULTIPLIER (Quality boost/penalty)
        # ====================================================================
        rating = ing_info.get('rating')
        if rating in RATING_MULTIPLIERS:
            ingredient_value *= RATING_MULTIPLIERS[rating]
            
            # Flag bad/worst ingredients as risks
            if rating in ['Bad', 'Worst']:
                has_risks = True
        
        total_score += ingredient_value
    
    # Normalize by number of ingredients
    if len(product['ingredient_list']) > 0:
        total_score /= len(product['ingredient_list'])
    
    return total_score, has_risks

print('✅ Enhanced ingredient scoring with ratings!')

✅ Enhanced ingredient scoring with ratings!


## Part 3: Enhanced Filtering Functions

In [7]:
def has_ingredient(product, ingredient_name):
    """Check if product contains specific ingredient"""
    if not isinstance(product['ingredient_list'], np.ndarray):
        return False
    
    search_term = ingredient_name.lower()
    return any(search_term in ing.lower() for ing in product['ingredient_list'])

def filter_by_required_ingredients(candidates, required_ingredients):
    """Keep only products that contain ALL required ingredients"""
    for req_ing in required_ingredients:
        candidates = candidates[candidates.apply(lambda x: has_ingredient(x, req_ing), axis=1)]
    return candidates

def filter_by_blocked_ingredients(candidates, blocked_ingredients):
    """Remove products that contain ANY blocked ingredients"""
    for blocked_ing in blocked_ingredients:
        candidates = candidates[~candidates.apply(lambda x: has_ingredient(x, blocked_ing), axis=1)]
    return candidates

print('✅ Ingredient filtering functions ready')

✅ Ingredient filtering functions ready


## Part 4: Explainability Function

In [8]:
def explain_recommendation(product, user_query, ingredient_db):
    """
    Generate explanation for why product was recommended
    
    Returns:
        dict with matched_concerns, key_ingredients, warnings
    """
    explanation = {
        'matched_concerns': [],
        'key_ingredients': [],
        'warnings': []
    }
    
    user_concerns = set(user_query.get('concerns', []))
    
    # 1. Show matched concerns from product labels
    product_concerns = set(product['positive_concerns']) if isinstance(product['positive_concerns'], np.ndarray) else set()
    explanation['matched_concerns'] = list(user_concerns & product_concerns)
    
    # 2. Show key ingredients that address user concerns
    if isinstance(product['ingredient_list'], np.ndarray):
        for ing in product['ingredient_list'][:15]:  # Top 15 ingredients
            ing_lower = ing.lower()
            if ing_lower in ingredient_db:
                ing_info = ingredient_db[ing_lower]
                
                # Check if ingredient matches any concern
                for concern in user_concerns:
                    matched = False
                    
                    # Check categories
                    if concern in CONCERN_TO_CATEGORY:
                        for cat in ing_info.get('categories', []):
                            if cat in CONCERN_TO_CATEGORY[concern]:
                                explanation['key_ingredients'].append(
                                    f"{ing_info['name']} ({cat}) → addresses '{concern}'"
                                )
                                matched = True
                                break
                    
                    if matched:
                        break
    
    # 3. Show warnings from product labels
    if isinstance(product['negative_concerns'], np.ndarray):
        for neg in product['negative_concerns']:
            explanation['warnings'].append(f"May cause {neg.replace('_', ' ')}")
    
    if isinstance(product['condition_concerns'], np.ndarray):
        for cond in product['condition_concerns']:
            explanation['warnings'].append(f"May worsen {cond}")
    
    # 4. Show ingredient-level risks
    if isinstance(product['ingredient_list'], np.ndarray):
        risky_ingredients = []
        for ing in product['ingredient_list']:
            ing_lower = ing.lower()
            if ing_lower in ingredient_db:
                ing_info = ingredient_db[ing_lower]
                categories = ing_info.get('categories', [])
                
                if 'Irritant' in categories:
                    risky_ingredients.append(f"{ing_info['name']} (irritant)")
                if 'Fragrance: Synthetic and Natural' in categories:
                    risky_ingredients.append(f"{ing_info['name']} (fragrance)")
        
        if risky_ingredients:
            explanation['warnings'].append(
                f"Contains: {', '.join(risky_ingredients[:3])}"  # Show first 3
            )
    
    return explanation

print('✅ Explainability function ready')

✅ Explainability function ready


In [9]:
def print_recommendation_ui(rec, query, ingredient_lookup):
    """
    Production-ready recommendation display
    Optimized for user interface presentation
    """
    
    print(f"\n{'='*80}")
    print(f"RANK #{rec['rank']}")
    print('='*80)
    
    # ========================================================================
    # SECTION 1: PRODUCT INFO
    # ========================================================================
    print(f"\n📦 {rec['name']}")
    print(f"   {rec['brand']} • {rec['type']} • {rec.get('country', 'Unknown')}")
    
    # ========================================================================
    # SECTION 2: MATCH SCORE
    # ========================================================================
    print(f"\n⭐ MATCH SCORE: {rec['score']*100:.0f}%")
    print(f"   └─ Based on your concerns ({rec['concern_sim']*100:.0f}%) and ingredient quality ({rec['ingredient_score']*100:.0f}%)")
    
    # ========================================================================
    # SECTION 3: WHY THIS PRODUCT? (Grouped by concern)
    # ========================================================================
    explanation = explain_recommendation(rec, query, ingredient_lookup)
    
    if explanation['matched_concerns'] or explanation['key_ingredients']:
        print(f"\n✅ WHY THIS PRODUCT MATCHES:")
        
        # Group ingredients by concern
        concern_to_ingredients = {}
        user_concerns = set(query.get('concerns', []))
        
        if isinstance(rec['ingredient_list'], np.ndarray):
            for ing in rec['ingredient_list'][:20]:  # Top 20 ingredients
                ing_lower = ing.lower()
                if ing_lower in ingredient_lookup:
                    ing_info = ingredient_lookup[ing_lower]
                    
                    for concern in user_concerns:
                        # Check categories
                        if concern in CONCERN_TO_CATEGORY:
                            for cat in ing_info.get('categories', []):
                                if cat in CONCERN_TO_CATEGORY[concern]:
                                    if concern not in concern_to_ingredients:
                                        concern_to_ingredients[concern] = []
                                    
                                    # Add rating to display
                                    rating = ing_info.get('rating', '')
                                    rating_display = f" [{rating}]" if rating else ""
                                    concern_to_ingredients[concern].append(f"{ing_info['name']} ({cat}){rating_display}")
                                    break
                        
                        # Check benefits
                        if concern in CONCERN_TO_BENEFIT:
                            for ben in ing_info.get('benefits', []):
                                if ben in CONCERN_TO_BENEFIT[concern]:
                                    if concern not in concern_to_ingredients:
                                        concern_to_ingredients[concern] = []
                                    if ing_info['name'] not in str(concern_to_ingredients[concern]):
                                        rating = ing_info.get('rating', '')
                                        rating_display = f" [{rating}]" if rating else ""
                                        concern_to_ingredients[concern].append(f"{ing_info['name']} ({ben}){rating_display}")
                                    break
        
        # Display grouped by concern
        for concern in user_concerns:
            concern_display = concern.replace('_', ' ').title()
            if concern in concern_to_ingredients:
                ingredients = concern_to_ingredients[concern][:5]  # Top 5 per concern
                print(f"\n   {concern_display}:")
                print(f"      → {', '.join(ingredients)}")
            elif concern in explanation['matched_concerns']:
                print(f"\n   {concern_display}:")
                print(f"      → Product labeled for this concern")
    
    # ========================================================================
    # SECTION 4: PRODUCT CLAIMS (What manufacturer says)
    # ========================================================================
    print(f"\n📋 MANUFACTURER CLAIMS:")
    
    if isinstance(rec['positive_concerns'], np.ndarray) and len(rec['positive_concerns']) > 0:
        claims = [c.replace('_', ' ').title() for c in rec['positive_concerns']]
        print(f"   Good for: {', '.join(claims)}")
    
    if isinstance(rec['negative_concerns'], np.ndarray) and len(rec['negative_concerns']) > 0:
        warnings = [c.replace('_', ' ') for c in rec['negative_concerns']]
        print(f"   May cause: {', '.join(warnings)}")
    
    if isinstance(rec['condition_concerns'], np.ndarray) and len(rec['condition_concerns']) > 0:
        conditions = [c.title() for c in rec['condition_concerns']]
        print(f"   Avoid if you have: {', '.join(conditions)}")
    
    # ========================================================================
    # SECTION 5: WARNINGS (Safety alerts)
    # ========================================================================
    if explanation['warnings']:
        print(f"\n⚠️  SAFETY WARNINGS:")
        for warning in explanation['warnings']:
            print(f"   • {warning}")
    else:
        print(f"\n✅ NO SAFETY WARNINGS - Safe for your skin type!")
    
    # ========================================================================
    # SECTION 6: VERIFICATION (User requirements check)
    # ========================================================================

    # Helper function for fragrance detection
    def has_fragrance_ingredients(product, ingredient_db):
        """Check if product contains ANY fragrance ingredients (by category)"""
        if not isinstance(product['ingredient_list'], np.ndarray):
            return False
        
        for ing in product['ingredient_list']:
            ing_lower = ing.lower()
            if ing_lower in ingredient_db:
                ing_info = ingredient_db[ing_lower]
                categories = ing_info.get('categories', [])
                if 'Fragrance: Synthetic and Natural' in categories:
                    return True
        return False

    # Build verification list
    verifications = []

    # Check required ingredients
    if 'required_ingredients' in query and query['required_ingredients']:
        for req_ing in query['required_ingredients']:
            has_it = any(req_ing.lower() in ing.lower() for ing in rec['ingredient_list'])
            status = '✅ YES' if has_it else '❌ MISSING'
            verifications.append((f"Contains {req_ing.title()}", status))

    # Check blocked ingredients - USE INGREDIENT DATABASE FOR SMART CHECKING
    if 'blocked_ingredients' in query and query['blocked_ingredients']:
        for blocked_ing in query['blocked_ingredients']:
            
            # Special case 1: "fragrance" - check by CATEGORY not name
            if blocked_ing.lower() == 'fragrance':
                has_fragrance = has_fragrance_ingredients(rec, ingredient_lookup)
                status = '❌ FOUND' if has_fragrance else '✅ FREE'
                verifications.append(("Fragrance-free", status))
            
            # Special case 2: "alcohol" - check for DRYING alcohols only
            elif blocked_ing.lower() == 'alcohol':
                # Bad alcohols (drying): Alcohol Denat, SD Alcohol, Isopropyl Alcohol
                # Good alcohols (not drying): Cetyl Alcohol, Stearyl Alcohol, Cetearyl Alcohol
                bad_alcohols = ['alcohol denat', 'sd alcohol', 'isopropyl alcohol', 'denatured alcohol']
                has_bad_alcohol = False
                
                if isinstance(rec['ingredient_list'], np.ndarray):
                    for ing in rec['ingredient_list']:
                        ing_lower = ing.lower()
                        # Check if it contains bad alcohol but NOT good alcohol
                        if any(bad_alc in ing_lower for bad_alc in bad_alcohols):
                            # Make sure it's not a fatty alcohol (good)
                            good_alcohols = ['cetyl', 'stearyl', 'cetearyl', 'behenyl']
                            if not any(good_alc in ing_lower for good_alc in good_alcohols):
                                has_bad_alcohol = True
                                break
                
                status = '❌ FOUND' if has_bad_alcohol else '✅ FREE'
                verifications.append(("Drying alcohol-free", status))
            
            # Regular ingredient name check for other ingredients
            else:
                has_it = any(blocked_ing.lower() in ing.lower() for ing in rec['ingredient_list'])
                status = '❌ FOUND' if has_it else '✅ FREE'
                verifications.append((f"{blocked_ing.title()}-free", status))

    # Check skin type safety
    skin_type = query.get('skin_type')
    if skin_type and isinstance(rec['negative_concerns'], np.ndarray):
        if skin_type == 'dry_skin':
            is_safe = 'drying' not in rec['negative_concerns']
            verifications.append(("Safe for dry skin", '✅ YES' if is_safe else '⚠️ MAY DRY'))
        elif skin_type == 'oily_skin':
            is_safe = 'may_worsen_oily_skin' not in rec['negative_concerns']
            verifications.append(("Safe for oily skin", '✅ YES' if is_safe else '⚠️ MAY WORSEN'))
        elif skin_type == 'sensitive_skin':
            is_safe = 'irritating' not in rec['negative_concerns']
            verifications.append(("Safe for sensitive skin", '✅ YES' if is_safe else '⚠️ MAY IRRITATE'))
        elif skin_type == 'combination_skin':
            conflicts = []
            if 'drying' in rec['negative_concerns']:
                conflicts.append('drying')
            if 'may_worsen_oily_skin' in rec['negative_concerns']:
                conflicts.append('oily')
            
            if len(conflicts) == 0:
                verifications.append(("Safe for combination skin", '✅ YES'))
            else:
                verifications.append(("Safe for combination skin", f"⚠️ MAY BE {'/'.join(conflicts).upper()}"))

    # Display verifications
    if verifications:
        print(f"\n🔍 VERIFICATION:")
        for check, status in verifications:
            print(f"   {check:<35} {status}")
        
    # ========================================================================
    # SECTION 7: FULL INGREDIENT LIST (Collapsible in UI)
    # ========================================================================
    if isinstance(rec['ingredient_list'], np.ndarray):
        print(f"\n📝 INGREDIENTS ({len(rec['ingredient_list'])} total):")
        # Single line, comma-separated (easy to copy)
        print(f"   {', '.join(rec['ingredient_list'])}")

print('✅ Production-ready UI function added!')

✅ Production-ready UI function added!


## Part 5: Hybrid Recommender (Full Features)

In [10]:
def recommend_hybrid_intelligent(user_query, penalty_value=0.9, ingredient_intelligence_weight=0.2, n=10):
    """
    Enhanced hybrid recommendation system
    
    NEW Features:
    - required_ingredients: list of must-have ingredients (e.g., ['niacinamide'])
    - blocked_ingredients: list of must-avoid ingredients (e.g., ['fragrance', 'alcohol'])
    - Ingredient intelligence scoring with risk detection
    
    Parameters:
    - penalty_value: Skin type penalty multiplier (0.8-1.0)
    - ingredient_intelligence_weight: Balance between concerns (0) and ingredients (1)
    """
    
    # Step 1: Product type filtering
    product_type = user_query.get('product_type', 'all')
    if product_type != 'all':
        candidates = products[products['type'] == product_type].copy()
    else:
        candidates = products.copy()
    
    if len(candidates) == 0:
        return pd.DataFrame()
    
    # Step 2: Hard safety filtering (concern/condition conflicts)
    avoid = set()
    for concern in user_query.get('concerns', []):
        if concern in CONCERN_RULES:
            avoid.update(CONCERN_RULES[concern])
    
    user_conditions = user_query.get('skin_conditions', [])
    
    def is_safe(row):
        neg = row['negative_concerns']
        if isinstance(neg, np.ndarray):
            for concern in avoid:
                if concern in neg:
                    return False
        
        cond = row['condition_concerns']
        if isinstance(cond, np.ndarray):
            for condition in user_conditions:
                if condition in cond:
                    return False
        return True
    
    candidates = candidates[candidates.apply(is_safe, axis=1)]
    
    if len(candidates) == 0:
        return pd.DataFrame()
    
    # Step 3: NEW - Required ingredient filtering
    if 'required_ingredients' in user_query and user_query['required_ingredients']:
        candidates = filter_by_required_ingredients(candidates, user_query['required_ingredients'])
    
    # Step 4: NEW - Blocked ingredient filtering
    if 'blocked_ingredients' in user_query and user_query['blocked_ingredients']:
        candidates = filter_by_blocked_ingredients(candidates, user_query['blocked_ingredients'])
    
    if len(candidates) == 0:
        return pd.DataFrame()
    
    # Step 5: Calculate concern-based similarity
    candidate_indices = candidates.index.tolist()
    candidate_features = features[candidate_indices]
    query_vector = create_query_vector(user_query)
    concern_similarities = cosine_similarity(query_vector.reshape(1, -1), candidate_features)[0]
    
    # Step 6: Calculate ingredient-based scores
    user_concerns = set(user_query.get('concerns', []))
    ingredient_scores = []
    ingredient_risks = []
    
    for idx in candidate_indices:
        score, has_risks = calculate_ingredient_score(candidates.loc[idx], user_concerns, ingredient_lookup)
        ingredient_scores.append(score)
        ingredient_risks.append(has_risks)
    
    ingredient_scores = np.array(ingredient_scores)
    
    # Normalize ingredient scores to 0-1 range
    if ingredient_scores.max() > 0:
        ingredient_scores = ingredient_scores / ingredient_scores.max()
    
    # Step 7: Hybrid scoring
    hybrid_scores = (concern_similarities * (1 - ingredient_intelligence_weight)) + \
                   (ingredient_scores * ingredient_intelligence_weight)
    
    # Step 8: Soft penalty for skin type conflicts
    skin_type = user_query.get('skin_type')
    if skin_type and skin_type in SKIN_TYPE_PENALTY_RULES:
        penalty_tags = SKIN_TYPE_PENALTY_RULES[skin_type]
        
        for i, product_idx in enumerate(candidate_indices):
            product = products.loc[product_idx]
            neg = product['negative_concerns']
            
            if isinstance(neg, np.ndarray):
                penalty_count = sum(1 for tag in penalty_tags if tag in neg)
                if penalty_count > 0:
                    hybrid_scores[i] *= (penalty_value ** penalty_count)
    
    # Step 9: Get top N
    top_n = min(n, len(candidates))
    top_indices = np.argsort(hybrid_scores)[::-1][:top_n]
    
    results = candidates.iloc[top_indices].copy()
    results['score'] = hybrid_scores[top_indices]
    results['concern_sim'] = concern_similarities[top_indices]
    results['ingredient_score'] = ingredient_scores[top_indices]
    results['has_ingredient_risks'] = [ingredient_risks[i] for i in top_indices]
    results['rank'] = range(1, len(results) + 1)
    
    return results

print('✅ Enhanced hybrid recommender ready')

✅ Enhanced hybrid recommender ready


## Part 6: Improved Evaluation (Fixed Safety Metric)

In [11]:
def eval_query_comprehensive(query, penalty, ingredient_intelligence_weight):
    """
    FIXED: Comprehensive evaluation including ingredient-level safety
    
    Safety now checks:
    1. Skin type conflicts (from product labels)
    2. Ingredient-level risks (irritants, fragrance)
    """
    user_wants = set(query['concerns'])
    recs = recommend_hybrid_intelligent(query, penalty_value=penalty, ingredient_intelligence_weight=ingredient_intelligence_weight, n=10)
    
    if len(recs) == 0:
        return None
    
    # Calculate NDCG
    relevance = []
    for _, row in recs.iterrows():
        prod_concerns = set(row['positive_concerns']) if isinstance(row['positive_concerns'], np.ndarray) else set()
        overlap = len(user_wants & prod_concerns)
        relevance.append(overlap)
    
    ndcg = 0
    if sum(relevance) > 0:
        ndcg = ndcg_score([relevance], [list(recs['score'])], k=10)
    
    # Calculate comprehensive safety
    safe_count = 0
    total_count = len(recs)
    
    skin_type = query.get('skin_type')
    
    for _, row in recs.iterrows():
        is_safe = True
        
        # Check 1: Skin type conflicts (product labels)
        if skin_type:
            neg = row['negative_concerns']
            if isinstance(neg, np.ndarray):
                if skin_type == 'dry_skin' and 'drying' in neg:
                    is_safe = False
                elif skin_type == 'oily_skin' and 'may_worsen_oily_skin' in neg:
                    is_safe = False
                elif skin_type == 'sensitive_skin' and 'irritating' in neg:
                    is_safe = False
        
        # Check 2: Ingredient-level risks (NEW!)
        if row.get('has_ingredient_risks', False):
            is_safe = False
        
        if is_safe:
            safe_count += 1
    
    safety = safe_count / total_count if total_count > 0 else 0
    
    return {'ndcg': ndcg, 'safety': safety}

print('✅ Comprehensive evaluation function ready')

✅ Comprehensive evaluation function ready


## Part 7: Quick Feature Test

In [12]:
print('='*80)
print('FEATURE TEST: Required/Blocked Ingredients')
print('='*80)

test_query = {
    'product_type': 'serum',
    'concerns': ['hydrating', 'anti_aging'],
    'skin_type': 'sensitive_skin',
    'required_ingredients': ['niacinamide'],  # Must have
    'blocked_ingredients': ['fragrance']      # Must not have
}

print(f'Query: {test_query}\n')

recs = recommend_hybrid_intelligent(test_query, penalty_value=0.9, ingredient_intelligence_weight=0.2, n=5)

if len(recs) > 0:
    print(f'Found {len(recs)} products matching ALL criteria:\n')
    
    for _, rec in recs.iterrows():
        print(f"Rank {rec['rank']}: {rec['name'][:60]}")
        
        # Verify niacinamide present
        has_nia = any('niacinamide' in ing.lower() for ing in rec['ingredient_list'])
        print(f"  ✓ Contains niacinamide: {has_nia}")
        
        # Verify no fragrance
        has_frag = any('fragrance' in ing.lower() for ing in rec['ingredient_list'])
        print(f"  ✓ Fragrance-free: {not has_frag}")
        print()
else:
    print('No products match these strict requirements!')
    print('Try relaxing some filters.')

FEATURE TEST: Required/Blocked Ingredients
Query: {'product_type': 'serum', 'concerns': ['hydrating', 'anti_aging'], 'skin_type': 'sensitive_skin', 'required_ingredients': ['niacinamide'], 'blocked_ingredients': ['fragrance']}

Found 5 products matching ALL criteria:

Rank 1: Bifida Complex Repair
  ✓ Contains niacinamide: True
  ✓ Fragrance-free: True

Rank 2: TW-Real Bifida Ampoule
  ✓ Contains niacinamide: True
  ✓ Fragrance-free: True

Rank 3: Skin Booster Vitamin Shot Time Restoring
  ✓ Contains niacinamide: True
  ✓ Fragrance-free: True

Rank 4: Skin Booster Vitamin Shot Brightening
  ✓ Contains niacinamide: True
  ✓ Fragrance-free: True

Rank 5: Niacinamide 20 Ampoule
  ✓ Contains niacinamide: True
  ✓ Fragrance-free: True



## Part 8: Hyperparameter Optimization

In [13]:
# Train/test split
train_queries, test_queries = train_test_split(all_queries, test_size=0.4, random_state=42)

print(f'Training queries: {len(train_queries)}')
print(f'Test queries: {len(test_queries)}')

Training queries: 120
Test queries: 80


In [14]:
print('='*80)
print('GRID SEARCH: penalty × ingredient_intelligence_weight')
print('='*80)

param_grid = {
    'penalty': [0.8, 0.9, 1.0],
    'ingredient_intelligence_weight': [0.0, 0.1, 0.2, 0.3, 0.4, 0.5]
}

kf = KFold(n_splits=5, shuffle=True, random_state=42)
results = []

for penalty in param_grid['penalty']:
    for ing_intel_weight in param_grid['ingredient_intelligence_weight']:
        
        fold_ndcgs = []
        fold_safeties = []
        
        for train_idx, val_idx in kf.split(train_queries):
            val_queries = [train_queries[i] for i in val_idx]
            
            metrics = [eval_query_comprehensive(q, penalty, ing_intel_weight) for q in val_queries]
            metrics = [m for m in metrics if m is not None]
            
            if len(metrics) > 0:
                fold_ndcgs.append(np.mean([m['ndcg'] for m in metrics]))
                fold_safeties.append(np.mean([m['safety'] for m in metrics]))
        
        avg_ndcg = np.mean(fold_ndcgs) if fold_ndcgs else 0
        avg_safety = np.mean(fold_safeties) if fold_safeties else 0
        
        results.append({
            'penalty': penalty,
            'ingredient_intelligence_weight': ing_intel_weight,
            'ndcg': avg_ndcg,
            'safety': avg_safety
        })
        
        print(f'penalty={penalty:.1f}, ing_intel={ing_intel_weight:.1f} → NDCG={avg_ndcg:.4f}, Safety={avg_safety:.1%}')

results_df = pd.DataFrame(results)

# Find best (maximize NDCG with safety >= 70%)
safe_results = results_df[results_df['safety'] >= 0.70]
if len(safe_results) > 0:
    best = safe_results.nlargest(1, 'ndcg').iloc[0]
else:
    best = results_df.nlargest(1, 'ndcg').iloc[0]

print('\n' + '='*80)
print('BEST PARAMETERS')
print('='*80)
print(f"penalty: {best['penalty']:.1f}")
print(f"ingredient_intelligence_weight: {best['ingredient_intelligence_weight']:.1f}")
print(f"NDCG: {best['ndcg']:.4f}")
print(f"Safety: {best['safety']:.1%}")

GRID SEARCH: penalty × ingredient_intelligence_weight
penalty=0.8, ing_intel=0.0 → NDCG=0.9649, Safety=40.8%
penalty=0.8, ing_intel=0.1 → NDCG=0.9667, Safety=47.1%
penalty=0.8, ing_intel=0.2 → NDCG=0.9689, Safety=50.7%
penalty=0.8, ing_intel=0.3 → NDCG=0.9657, Safety=55.4%
penalty=0.8, ing_intel=0.4 → NDCG=0.9551, Safety=58.7%
penalty=0.8, ing_intel=0.5 → NDCG=0.9370, Safety=62.4%
penalty=0.9, ing_intel=0.0 → NDCG=0.9661, Safety=38.9%
penalty=0.9, ing_intel=0.1 → NDCG=0.9677, Safety=46.3%
penalty=0.9, ing_intel=0.2 → NDCG=0.9689, Safety=49.5%
penalty=0.9, ing_intel=0.3 → NDCG=0.9656, Safety=53.8%
penalty=0.9, ing_intel=0.4 → NDCG=0.9555, Safety=56.6%
penalty=0.9, ing_intel=0.5 → NDCG=0.9376, Safety=60.1%
penalty=1.0, ing_intel=0.0 → NDCG=0.9654, Safety=38.1%
penalty=1.0, ing_intel=0.1 → NDCG=0.9696, Safety=45.6%
penalty=1.0, ing_intel=0.2 → NDCG=0.9707, Safety=47.7%
penalty=1.0, ing_intel=0.3 → NDCG=0.9688, Safety=51.8%
penalty=1.0, ing_intel=0.4 → NDCG=0.9601, Safety=53.6%
penalty=1.0

## Part 9: Test Set Evaluation

In [15]:
print('='*80)
print('TEST SET EVALUATION')
print('='*80)
print(f"Using best: penalty={best['penalty']:.1f}, ing_intel_weight={best['ingredient_intelligence_weight']:.1f}\n")

# Evaluate on test set
test_metrics = [eval_query_comprehensive(q, best['penalty'], best['ingredient_intelligence_weight']) for q in test_queries]
test_metrics = [m for m in test_metrics if m is not None]

test_ndcg = np.mean([m['ndcg'] for m in test_metrics])
test_safety = np.mean([m['safety'] for m in test_metrics])

# Baseline (ing_intel_weight=0.0)
baseline_metrics = [eval_query_comprehensive(q, best['penalty'], 0.0) for q in test_queries]
baseline_metrics = [m for m in baseline_metrics if m is not None]

baseline_ndcg = np.mean([m['ndcg'] for m in baseline_metrics])
baseline_safety = np.mean([m['safety'] for m in baseline_metrics])

print('='*80)
print('COMPARISON: Baseline vs Ingredient-Aware (FIXED SAFETY)')
print('='*80)
print(f'{"Metric":<25} {"Baseline":<20} {"Ingredient-Aware":<20} {"Change"}')
print('-'*80)
print(f'{"NDCG@10":<25} {baseline_ndcg:.4f}{"":<15} {test_ndcg:.4f}{"":<15} {(test_ndcg - baseline_ndcg):+.4f}')
print(f'{"Safety (comprehensive)":<25} {baseline_safety:.1%}{"":<15} {test_safety:.1%}{"":<15} {(test_safety - baseline_safety):+.1%}')

if test_ndcg > baseline_ndcg and test_safety >= baseline_safety:
    print(f'\n✅ Ingredient intelligence IMPROVED both NDCG and Safety!')
elif test_ndcg > baseline_ndcg:
    print(f'\n✅ Ingredient intelligence IMPROVED NDCG by {(test_ndcg - baseline_ndcg)*100:.2f}%')
elif test_safety > baseline_safety:
    print(f'\n✅ Ingredient intelligence IMPROVED Safety by {(test_safety - baseline_safety)*100:.1f}%')
else:
    print(f'\n→ Marginal differences - both systems perform similarly')

TEST SET EVALUATION
Using best: penalty=1.0, ing_intel_weight=0.2

COMPARISON: Baseline vs Ingredient-Aware (FIXED SAFETY)
Metric                    Baseline             Ingredient-Aware     Change
--------------------------------------------------------------------------------
NDCG@10                   0.9781                0.9782                +0.0001
Safety (comprehensive)    38.6%                49.5%                +10.9%

✅ Ingredient intelligence IMPROVED both NDCG and Safety!


## Part 10: Demo with Explainability

In [16]:
demo_query = {
    'product_type': 'serum',
    'concerns': ['hydrating', 'anti_aging'],
    'skin_type': 'sensitive_skin'
}

print('='*80)
print('DEMO: Recommendations with Explanation')
print('='*80)
print(f'Query: {demo_query}\n')

recs = recommend_hybrid_intelligent(
    demo_query,
    penalty_value=best['penalty'],
    ingredient_intelligence_weight=best['ingredient_intelligence_weight'],
    n=3
)

for _, rec in recs.iterrows():
    print('\n' + '='*80)
    print(f"RANK {rec['rank']}: {rec['name']}")
    print('='*80)
    print(f"Brand: {rec['brand']}")
    print(f"Type: {rec['type']}")
    print(f"\nScoring:")
    print(f"  Final Score:        {rec['score']:.3f}")
    print(f"  ├─ Concern match:   {rec['concern_sim']:.3f}")
    print(f"  └─ Ingredient intel: {rec['ingredient_score']:.3f}")
    
    # Get explanation
    explanation = explain_recommendation(rec, demo_query, ingredient_lookup)
    
    if explanation['matched_concerns']:
        print(f"\n✅ ADDRESSES YOUR CONCERNS:")
        for concern in explanation['matched_concerns']:
            print(f"   • {concern.replace('_', ' ').title()}")
    
    if explanation['key_ingredients']:
        print(f"\n🔬 KEY ACTIVE INGREDIENTS:")
        for ing_desc in explanation['key_ingredients'][:5]:  # Top 5
            print(f"   • {ing_desc}")
    
    if explanation['warnings']:
        print(f"\n⚠️  WARNINGS:")
        for warning in explanation['warnings']:
            print(f"   • {warning}")
    else:
        print(f"\n✅ No safety warnings for your skin type!")

DEMO: Recommendations with Explanation
Query: {'product_type': 'serum', 'concerns': ['hydrating', 'anti_aging'], 'skin_type': 'sensitive_skin'}


RANK 1: Vegan Collagen Booster Serum
Brand: boscia
Type: serum

Scoring:
  Final Score:        0.901
  ├─ Concern match:   1.000
  └─ Ingredient intel: 0.507

✅ ADDRESSES YOUR CONCERNS:
   • Anti Aging
   • Hydrating

🔬 KEY ACTIVE INGREDIENTS:
   • Pentylene Glycol (Humectant) → addresses 'hydrating'
   • Glycerin (Humectant) → addresses 'hydrating'
   • Collagen (Humectant) → addresses 'hydrating'
   • Emblica Officinalis Fruit Extract (Antioxidant) → addresses 'anti_aging'
   • Tripeptide-29 (Peptides) → addresses 'anti_aging'

✅ No safety warnings for your skin type!

RANK 2: Collagen Serum
Brand: Skin Inc
Type: serum

Scoring:
  Final Score:        0.897
  ├─ Concern match:   1.000
  └─ Ingredient intel: 0.486

✅ ADDRESSES YOUR CONCERNS:
   • Anti Aging
   • Hydrating

🔬 KEY ACTIVE INGREDIENTS:
   • Butylene Glycol (Humectant) → addresses

## Part 11: Ingredient Quality & Rating Demo

In [17]:
# ============================================================================
# DEMO 1: User Selects "Antioxidant" Concern
# ============================================================================

query_antioxidant = {
    'product_type': 'serum',
    'concerns': ['antioxidant', 'hydrating'],  # User picks from dropdown
    'skin_type': 'normal_skin'
}

print('='*80)
print('DEMO 1: User Selects "Antioxidant" Concern')
print('='*80)
print(f'Query: {query_antioxidant}')
print('System automatically finds products with antioxidant ingredients...\n')

recs = recommend_hybrid_intelligent(
    query_antioxidant,
    penalty_value=best['penalty'],
    ingredient_intelligence_weight=0.3,  # Higher focus on ingredients
    n=3
)

for _, rec in recs.iterrows():
    print('\n' + '='*80)
    print(f"RANK #{rec['rank']}: {rec['name']}")
    print('='*80)
    print(f"Brand: {rec['brand']}")
    print(f"\n⭐ MATCH SCORE: {rec['score']*100:.0f}%")
    print(f"   ├─ Concern: {rec['concern_sim']*100:.0f}%")
    print(f"   └─ Ingredient quality: {rec['ingredient_score']*100:.0f}%")
    
    # Find antioxidants
    print(f"\n🛡️  ANTIOXIDANT INGREDIENTS:")
    antioxidants = []
    
    if isinstance(rec['ingredient_list'], np.ndarray):
        for ing in rec['ingredient_list'][:15]:
            ing_lower = ing.lower()
            if ing_lower in ingredient_lookup:
                ing_info = ingredient_lookup[ing_lower]
                if 'Antioxidant' in ing_info.get('categories', []):
                    rating = ing_info.get('rating', 'Unknown')
                    antioxidants.append(f"{ing_info['name']} [{rating}]")
    
    if antioxidants:
        for antox in antioxidants[:5]:
            print(f"   • {antox}")
        if len(antioxidants) > 5:
            print(f"   ... and {len(antioxidants) - 5} more")
    
    # Find hydrating ingredients
    print(f"\n💧 HYDRATING INGREDIENTS:")
    hydrating = []
    
    if isinstance(rec['ingredient_list'], np.ndarray):
        for ing in rec['ingredient_list'][:15]:
            ing_lower = ing.lower()
            if ing_lower in ingredient_lookup:
                ing_info = ingredient_lookup[ing_lower]
                categories = ing_info.get('categories', [])
                if any(cat in ['Humectant', 'Emollient', 'Occlusive/Opacifying Agent'] for cat in categories):
                    if 'Antioxidant' not in categories:  # Don't double-count
                        rating = ing_info.get('rating', 'Unknown')
                        cat_name = [c for c in categories if c in ['Humectant', 'Emollient', 'Occlusive/Opacifying Agent']][0]
                        hydrating.append(f"{ing_info['name']} ({cat_name}) [{rating}]")
    
    if hydrating:
        for hyd in hydrating[:3]:
            print(f"   • {hyd}")


# ============================================================================
# DEMO 2: Ingredient Rating Impact
# ============================================================================

print('\n\n' + '='*80)
print('DEMO 2: How Ingredient Ratings Affect Recommendations')
print('='*80)
print('\nComparing products with SAME concerns but DIFFERENT ingredient quality...\n')

query_quality = {
    'product_type': 'serum',
    'concerns': ['anti_aging', 'brightening'],
    'skin_type': 'sensitive_skin'
}

recs_quality = recommend_hybrid_intelligent(
    query_quality,
    penalty_value=best['penalty'],
    ingredient_intelligence_weight=0.3,
    n=5
)

for _, rec in recs_quality.iterrows():
    print('='*80)
    print(f"RANK #{rec['rank']}: {rec['name'][:60]}")
    print('='*80)
    
    # Analyze ingredient quality distribution
    rating_counts = {'Best': 0, 'Good': 0, 'Average': 0, 'Bad': 0, 'Worst': 0}
    best_ingredients = []
    worst_ingredients = []
    
    if isinstance(rec['ingredient_list'], np.ndarray):
        for ing in rec['ingredient_list'][:20]:  # Top 20 ingredients
            ing_lower = ing.lower()
            if ing_lower in ingredient_lookup:
                ing_info = ingredient_lookup[ing_lower]
                rating = ing_info.get('rating')
                
                if rating in rating_counts:
                    rating_counts[rating] += 1
                    
                    if rating == 'Best':
                        best_ingredients.append(ing_info['name'])
                    elif rating in ['Bad', 'Worst']:
                        worst_ingredients.append(ing_info['name'])
    
    print(f"\n📊 INGREDIENT QUALITY BREAKDOWN:")
    print(f"   Best:    {rating_counts['Best']:2d} (×1.3 multiplier = 30% boost)")
    print(f"   Good:    {rating_counts['Good']:2d} (×1.1 multiplier = 10% boost)")
    print(f"   Average: {rating_counts['Average']:2d} (×1.0 multiplier = neutral)")
    print(f"   Bad:     {rating_counts['Bad']:2d} (×0.8 multiplier = 20% penalty)")
    print(f"   Worst:   {rating_counts['Worst']:2d} (×0.5 multiplier = 50% penalty)")
    
    if best_ingredients:
        print(f"\n🏆 PREMIUM INGREDIENTS:")
        for best_ing in best_ingredients[:3]:
            print(f"   • {best_ing}")
    
    if worst_ingredients:
        print(f"\n⚠️  PROBLEMATIC INGREDIENTS:")
        for worst_ing in worst_ingredients:
            print(f"   • {worst_ing}")
    else:
        print(f"\n✅ No problematic ingredients!")
    
    print()


# ============================================================================
# SUMMARY
# ============================================================================

print('\n' + '='*80)
print('💡 KEY INSIGHTS')
print('='*80)
print('''
WHY INGREDIENT RATINGS MATTER:

1. QUALITY DIFFERENTIATION:
   Two products can both claim "anti-aging + brightening", but:
   • Product A: Uses premium actives (Vitamin C derivative [Best], Retinol [Best])
   • Product B: Uses cheaper alternatives (Ascorbic Acid [Good], weak antioxidants [Average])
   → Product A ranks higher due to ingredient quality

2. SAFETY DETECTION:
   Ratings help identify problematic ingredients:
   • Best/Good: Safe, effective ingredients
   • Bad/Worst: Potential irritants, allergens, or ineffective ingredients
   → System automatically penalizes risky formulations

3. COMPLETE SCORING FORMULA:
   Ingredient Score = (
       Category Match (+1.0 per match)
       + Benefit Match (+0.5 bonus)
   ) × Functional Weight (Actives=2x, Risks=-1x)
     × Rating Multiplier (Best=1.3x, Worst=0.5x)

   Example:
   Vitamin C [Best, Active, matches antioxidant] = (1.0) × 2.0 × 1.3 = 2.6 points
   Fragrance [Worst, Sensory, negative] = (-1.0) × 0.25 × 0.5 = -0.125 points

This is why ingredient intelligence > just trusting product labels!
''')


DEMO 1: User Selects "Antioxidant" Concern
Query: {'product_type': 'serum', 'concerns': ['antioxidant', 'hydrating'], 'skin_type': 'normal_skin'}
System automatically finds products with antioxidant ingredients...


RANK #1: Resurfacing Amino Acid Serum
Brand: Acure

⭐ MATCH SCORE: 86%
   ├─ Concern: 100%
   └─ Ingredient quality: 53%

🛡️  ANTIOXIDANT INGREDIENTS:
   • Aloe Barbadensis Leaf Juice [Best]
   • Arginine [Best]
   • Serine [Good]
   • Alanine [Good]
   • Threonine [Good]
   ... and 2 more

💧 HYDRATING INGREDIENTS:
   • Betaine (Humectant) [Good]
   • Glycine (Humectant) [Best]
   • Proline (Humectant) [Best]

RANK #2: Facial Treatment Repair C
Brand: Sk-II

⭐ MATCH SCORE: 84%
   ├─ Concern: 100%
   └─ Ingredient quality: 46%

🛡️  ANTIOXIDANT INGREDIENTS:
   • Galactomyces Ferment Filtrate [Good]
   • Sodium Hyaluronate [Best]

💧 HYDRATING INGREDIENTS:
   • Butylene Glycol (Humectant) [Good]
   • Glycerin (Humectant) [Best]
   • Pentylene Glycol (Humectant) [Good]

RANK #3:

## Part 12: Production UI Demo (Full Display)

In [18]:
# ============================================================================
# PRODUCTION UI DEMO: Complete Recommendation Display
# ============================================================================

demo_query = {
    'product_type': 'serum',
    'concerns': ['antioxidant', 'hydrating', 'anti_aging'],
    'skin_type': 'sensitive_skin',
    'required_ingredients': ['niacinamide'],
    'blocked_ingredients': ['fragrance', 'alcohol']
}

print('='*80)
print('PRODUCTION UI DEMO: Complete Recommendation Display')
print('='*80)
print(f'\nUser Query:')
print(f'  Product Type: {demo_query["product_type"]}')
print(f'  Concerns: {", ".join(demo_query["concerns"])}')
print(f'  Skin Type: {demo_query["skin_type"]}')
print(f'  Must Have: {", ".join(demo_query["required_ingredients"])}')
print(f'  Must Avoid: {", ".join(demo_query["blocked_ingredients"])}')
print()

recs = recommend_hybrid_intelligent(
    demo_query,
    penalty_value=best['penalty'],
    ingredient_intelligence_weight=best['ingredient_intelligence_weight'],
    n=3
)

if len(recs) > 0:
    print(f'Found {len(recs)} products matching ALL requirements:\n')
    
    for _, rec in recs.iterrows():
        print_recommendation_ui(rec, demo_query, ingredient_lookup)
    
    print('\n' + '='*80)
    print('UI STRUCTURE EXPLANATION')
    print('='*80)
    print('''
Each recommendation shows 7 sections:

1. PRODUCT INFO
   - Product name, brand, type, country

2. MATCH SCORE
   - Overall percentage match
   - Breakdown: concern similarity + ingredient quality

3. WHY THIS PRODUCT MATCHES
   - Grouped by user concerns
   - Shows specific ingredients with ratings [Best/Good/etc]
   - Example: "Hydrating: Glycerin (Humectant) [Good], Hyaluronic Acid (Humectant) [Best]"

4. MANUFACTURER CLAIMS
   - What product labels say (positive, negative, conditions)
   - Transparency: shows both benefits AND warnings

5. SAFETY WARNINGS
   - Aggregated from both labels and ingredient analysis
   - Clear ⚠️ symbols for visibility

6. VERIFICATION
   - Confirms user requirements met
   - ✅/❌ status for each requirement
   - Skin type safety check

7. FULL INGREDIENT LIST
   - Complete INCI list
   - Easy to copy for further research
    ''')
    
else:
    print('❌ No products found matching ALL requirements!')
    print('\nSuggestions:')
    print('  • Try removing some blocked ingredients')
    print('  • Make some required ingredients optional')
    print('  • Relax skin type restrictions')


# ============================================================================
# COMPARISON: Short vs Full Display
# ============================================================================

print('\n\n' + '='*80)
print('FOR DEVELOPERS: Display Flexibility')
print('='*80)
print('''
This UI function is designed for PYTHON DEMOS and THESIS PRESENTATION.

For PRODUCTION (C# ML.NET API):
1. Return structured JSON with all fields
2. Let frontend choose what to display
3. Enable progressive disclosure (expand/collapse sections)

JSON Structure:
{
  "rank": 1,
  "product": {
    "name": "...",
    "brand": "...",
    "type": "..."
  },
  "scores": {
    "overall": 0.94,
    "concern_similarity": 0.98,
    "ingredient_quality": 0.87
  },
  "why_matched": {
    "hydrating": [
      {"ingredient": "Glycerin", "category": "Humectant", "rating": "Good"},
      {"ingredient": "Hyaluronic Acid", "category": "Humectant", "rating": "Best"}
    ],
    "antioxidant": [...]
  },
  "manufacturer_claims": {
    "positive": ["hydrating", "anti_aging"],
    "negative": ["drying"],
    "conditions": []
  },
  "warnings": [...],
  "verification": {
    "required_ingredients_met": true,
    "blocked_ingredients_absent": true,
    "safe_for_skin_type": true
  },
  "ingredients": ["Water", "Glycerin", ...]
}

Frontend can then:
• Show compact card view initially
• Expand to show full details on click
• Filter/sort by different criteria
• Highlight user-specified requirements
''')


PRODUCTION UI DEMO: Complete Recommendation Display

User Query:
  Product Type: serum
  Concerns: antioxidant, hydrating, anti_aging
  Skin Type: sensitive_skin
  Must Have: niacinamide
  Must Avoid: fragrance, alcohol

Found 3 products matching ALL requirements:


RANK #1

📦 Bifida Complex Repair
   Kranicell • serum • United States

⭐ MATCH SCORE: 59%
   └─ Based on your concerns (58%) and ingredient quality (66%)

✅ WHY THIS PRODUCT MATCHES:

   Antioxidant:
      → Galactomyces Ferment Filtrate (Antioxidant) [Good], Niacinamide (Antioxidant) [Best], Sodium Hyaluronate (Antioxidant) [Best], Panax Ginseng Root Extract (Antioxidant) [Best], Platycodon Grandiflorus Root Extract (Antioxidant) [Good]

   Anti Aging:
      → Galactomyces Ferment Filtrate (Antioxidant) [Good], Glycerin (Anti-Aging) [Best], Niacinamide (Antioxidant) [Best], Sodium Hyaluronate (Antioxidant) [Best], Panax Ginseng Root Extract (Antioxidant) [Best]

   Hydrating:
      → Butylene Glycol (Humectant) [Good], Gly

## Conclusion

### What We Built

Enhanced the concern-based recommender with **comprehensive ingredient intelligence**:

1. **Ingredient scoring**: Analyzes 4,985 ingredients across 3 dimensions (categories, benefits, functional groups)
2. **Advanced filtering**: Users can require specific ingredients or block unwanted ones
3. **Improved safety metric**: Checks both product labels AND ingredient-level risks
4. **Explainability**: Shows exactly why each product was recommended

### Results

See test set evaluation above for:
- Optimal `ingredient_intelligence_weight` (balance between labels and ingredients)
- NDCG improvement vs baseline
- Comprehensive safety rate (including ingredient risks)

### Key Insights

1. **Categories are most valuable**: 99.3% coverage vs 38.9% for benefits
2. **Balance is optimal**: Pure ingredient scoring ignores useful product labels
3. **Safety improved**: Fixed metric now catches ingredient-level risks (irritants, fragrance)
4. **User control matters**: Required/blocked ingredients enable precise filtering

### System Capabilities

**3-Layer Safety Architecture:**
- **Layer 1**: Hard filtering (removes dangerous products)
- **Layer 2**: Soft penalties (downranks risky products)
- **Layer 3**: Warnings (informs user of potential issues)

**Scoring Components:**
- Concern-based similarity (product labels)
- Ingredient intelligence (actual formulation)
- Functional weighting (Actives 2x, Risks -1x)

**User Controls:**
- Required ingredients ("must have retinol")
- Blocked ingredients ("no fragrance")
- Skin type preferences
- Concern priorities

### Limitations

- **Ingredient coverage**: Not all ingredients have detailed properties
- **No concentration data**: Can't distinguish 0.1% vs 2% active
- **Benefits sparsity**: Only 38.9% of ingredients have explicit benefits
- **Synthetic queries**: Need real user data for validation

### Production Deployment

**Immediate (Thesis Scope):**
1. Deploy as C# ML.NET API with tuned parameters
2. Implement explainability in UI (show why products recommended)
3. Add ingredient search filters (required/blocked)

**Future Enhancements:**
4. Expand ingredient database with concentration thresholds
5. Add comedogenic ratings for acne-prone users
6. Implement user feedback loop for personalization
7. A/B test ingredient-aware vs label-only recommendations